# 12 - Gercek Veri Baseline (is_missing + Median Imputation)

**Amac**: YARISMA_TRAIN_MASTER.csv uzerinde minimal on isleme ile baseline performansi olcmek.

## On Isleme Stratejisi

| NaN Orani | Strateji | Aciklama |
|-----------|----------|----------|
| > %50 | `is_missing_*` flag | Drop ETME — eksiklik bilgi tasiyor (Label=1'de %59.9, Label=0'da %41.2) |
| <= %50 | Median imputation | Median, aykiri deger iceren freakans/skor sutunlari icin mean'den daha robust |
| Kategorik | `MISSING` string | OHE/LE pipeline'da ayri kategori olarak islenir |

**Kritik kural**: Imputation, OHE ve LE fit'leri yalnizca **MASTER train** uzerinde yapilir.  
Test ve panellere transform uygulanir — veri sizintisi yok.

## Pipeline Yapisi

| Section | Encoding | Modeller |
|---------|----------|---------|
| A | One-Hot Encoding (OHE) | LightGBM, XGBoost, SVM |
| B | LabelEncoder (LE) | LightGBM, XGBoost, SVM |
| C | CatBoost native (OTS) | CatBoost |

**Split**: Hold-out %20, 3-fold CV grid search  
**Panel testi**: KANSER, PAH, CFTR -> MASTER modelleri ile dogrudan test

In [23]:
# Cell 1: Imports & Config
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, TEST_SIZE, REPORTS_DIR
from src.models import (
    grid_search_lightgbm, grid_search_xgboost, grid_search_svm,
    grid_search_le_lightgbm, grid_search_le_xgboost, grid_search_le_svm,
    grid_search_catboost,
)
from src.metrics import optimize_threshold, compute_all_metrics

REAL_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_data')
MASTER_PATH   = os.path.join(REAL_DATA_DIR, 'YARISMA_TRAIN_MASTER.csv')
PANEL_PATHS   = {
    'KANSER': os.path.join(REAL_DATA_DIR, 'YARISMA_TRAIN_KANSER.csv'),
    'PAH':    os.path.join(REAL_DATA_DIR, 'YARISMA_TRAIN_PAH.csv'),
    'CFTR':   os.path.join(REAL_DATA_DIR, 'YARISMA_TRAIN_CFTR.csv'),
}

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'real_data_baseline')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Yuksek eksik esigi: bu sutunlar is_missing flag'e donusur (drop edilmez)
HIGH_NAN_THRESHOLD = 0.50

print(f'Proje koku   : {PROJECT_ROOT}')
print(f'Sonuc dizini : {RESULTS_DIR}')

Proje koku   : /Users/tefe/teknofest_model/teknofest_model
Sonuc dizini : /Users/tefe/teknofest_model/teknofest_model/results/real_data_baseline


In [24]:
# Cell 2: Veri Yukleme & On Isleme (MASTER)
#
# Strateji:
#   - NaN > %50 olan sayisal sutunlar: degerler olduklari gibi KALIR
#     + ayrica `is_missing_<col>` = 0/1 flag sutunu EKLENIR
#   - NaN <= %50 olan sayisal sutunlar: MASTER train mediani ile doldurulur
#   - Kategorik sutunlar: NaN -> 'MISSING' string (OHE/LE ile islenir)
#
# NOT: median TRAIN ayrilmadan hesaplanmaz — asagida train split'ten sonra yapilir.
# Bu cell sadece ham analiz + sutun tespiti yapar.

df_master = pd.read_csv(MASTER_PATH)
print(f'MASTER ham veri: {df_master.shape}')
print(f'Label: {df_master["Label"].value_counts().to_dict()}')
print(f'Pozitif orani: {df_master["Label"].mean():.3f}')

# Variant_ID cikar, Label ayir
df_master = df_master.drop(columns=['Variant_ID'])
y_master  = df_master['Label'].astype(int)
X_master  = df_master.drop(columns=['Label'])

# Sutun turlerini belirle
nan_pct   = X_master.isnull().mean()
cat_cols  = [c for c in X_master.columns if X_master[c].dtype == object]
num_cols  = [c for c in X_master.columns if c not in cat_cols]

# Sayisal sutunlari nan oranina gore ikiye bol
high_nan_cols = [c for c in num_cols if nan_pct[c] > HIGH_NAN_THRESHOLD]
low_nan_cols  = [c for c in num_cols if nan_pct[c] <= HIGH_NAN_THRESHOLD]

print(f'\nKategorik sutun sayisi       : {len(cat_cols)} -> {cat_cols}')
print(f'Sayisal sutun (toplam)       : {len(num_cols)}')
print(f'  NaN > %50 (is_missing flag): {len(high_nan_cols)}')
print(f'  NaN <= %50 (median imp.)   : {len(low_nan_cols)}')
print(f'\nEklenen is_missing flag sayisi: {len(high_nan_cols)}')
print(f'Toplam ozellik (impute+flags): {len(num_cols) + len(high_nan_cols) + len(cat_cols)}')

MASTER ham veri: (2931, 353)
Label: {1: 2149, 0: 782}
Pozitif orani: 0.733

Kategorik sutun sayisi       : 8 -> ['CAT_1', 'CAT_2', 'CAT_3', 'CAT_4', 'CAT_5', 'CAT_6', 'AA_1', 'AA_2']
Sayisal sutun (toplam)       : 343
  NaN > %50 (is_missing flag): 163
  NaN <= %50 (median imp.)   : 180

Eklenen is_missing flag sayisi: 163
Toplam ozellik (impute+flags): 514


In [25]:
# Cell 3: Train/Test Split
# Split ONCE — sonra median/OHE/LE sadece X_train_raw uzerinde fit edilir.

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_master, y_master, test_size=TEST_SIZE, random_state=SEED, stratify=y_master
)

print(f'Train: {X_train_raw.shape}   Test: {X_test_raw.shape}')
print(f'Train label: {y_train.value_counts().to_dict()}')
print(f'Test  label: {y_test.value_counts().to_dict()}')

Train: (2344, 351)   Test: (587, 351)
Train label: {1: 1719, 0: 625}
Test  label: {1: 430, 0: 157}


In [26]:
# Cell 4: On Isleme Fonksiyonlari & MASTER Preprocessing
#
# preprocess_pipeline(X_train, X_test_or_panel, ...)
#   col_medians : sadece train'den fit (None ise hesapla, else kullan)
#   ohe         : sadece train'den fit (None ise fit et, else transform)
#
# Iki cikti matrisi uretilir:
#   X_ohe  : OHE + is_missing flags + median-imp numerics  (Section A + B)
#   X_raw  : ham str kategorik + is_missing flags + median-imp numerics (Section C CatBoost)

def add_missing_flags(X_df, high_nan_cols):
    """Yuksek NaN sutunlari icin is_missing_* binary flag ekle."""
    X = X_df.copy()
    for c in high_nan_cols:
        if c in X.columns:
            X[f'is_missing_{c}'] = X[c].isna().astype(np.float32)
    return X


def preprocess_ohe(
    X_train_df, X_other_df, cat_cols, low_nan_cols, high_nan_cols,
    col_medians=None, ohe=None
):
    """
    OHE pipeline: kategorikler binary sutunlara, sayisallar median-impute.
    Fit: yalnizca X_train_df (col_medians=None veya ohe=None ise).
    Returns: X_tr, X_ot, ohe, col_medians
    """
    X_tr = add_missing_flags(X_train_df, high_nan_cols)
    X_ot = add_missing_flags(X_other_df, high_nan_cols)
    all_num = low_nan_cols + [c for c in high_nan_cols if c in X_tr.columns]

    # Median imputation
    if col_medians is None:
        col_medians = X_tr[low_nan_cols].median()
    X_tr[low_nan_cols] = X_tr[low_nan_cols].fillna(col_medians)
    X_ot[low_nan_cols] = X_ot[low_nan_cols].fillna(col_medians)

    # high_nan sutunlarin kendi degerlerini de median ile doldur
    # (flag zaten eklendi; deger hala NaN kalirsa model icin 0 daha guvenli)
    high_medians = X_tr[high_nan_cols].median()
    X_tr[high_nan_cols] = X_tr[high_nan_cols].fillna(high_medians)
    X_ot[high_nan_cols] = X_ot[high_nan_cols].fillna(high_medians)

    # Kategorik: NaN -> 'MISSING'
    for c in cat_cols:
        X_tr[c] = X_tr[c].fillna('MISSING').astype(str)
        X_ot[c] = X_ot[c].fillna('MISSING').astype(str)

    # OHE
    if ohe is None:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.float32)
        ohe.fit(X_tr[cat_cols])

    ohe_names = ohe.get_feature_names_out(cat_cols)
    tr_ohe = pd.DataFrame(ohe.transform(X_tr[cat_cols]), columns=ohe_names, index=X_tr.index)
    ot_ohe = pd.DataFrame(ohe.transform(X_ot[cat_cols]), columns=ohe_names, index=X_ot.index)

    # Tum sayisal + is_missing + ohe
    flag_cols = [f'is_missing_{c}' for c in high_nan_cols if f'is_missing_{c}' in X_tr.columns]
    X_tr_out = pd.concat([X_tr[all_num + flag_cols].astype(np.float32), tr_ohe], axis=1)
    X_ot_out = pd.concat([X_ot[all_num + flag_cols].astype(np.float32), ot_ohe], axis=1)

    return X_tr_out, X_ot_out, ohe, col_medians, high_medians


def preprocess_raw(
    X_train_df, X_other_df, cat_cols, low_nan_cols, high_nan_cols,
    col_medians=None, high_medians=None
):
    """
    RAW pipeline (LE / CatBoost): kategorikler ham string kalir.
    Returns: X_tr, X_ot, col_medians, high_medians
    """
    X_tr = add_missing_flags(X_train_df, high_nan_cols)
    X_ot = add_missing_flags(X_other_df, high_nan_cols)

    if col_medians is None:
        col_medians = X_tr[low_nan_cols].median()
    X_tr[low_nan_cols] = X_tr[low_nan_cols].fillna(col_medians)
    X_ot[low_nan_cols] = X_ot[low_nan_cols].fillna(col_medians)

    if high_medians is None:
        high_medians = X_tr[high_nan_cols].median()
    X_tr[high_nan_cols] = X_tr[high_nan_cols].fillna(high_medians)
    X_ot[high_nan_cols] = X_ot[high_nan_cols].fillna(high_medians)

    for c in cat_cols:
        X_tr[c] = X_tr[c].fillna('MISSING').astype(str)
        X_ot[c] = X_ot[c].fillna('MISSING').astype(str)

    # Sutun sirasi: sayisal, is_missing flagler, kategorik
    all_num  = low_nan_cols + high_nan_cols
    flag_cols= [f'is_missing_{c}' for c in high_nan_cols if f'is_missing_{c}' in X_tr.columns]
    col_order = all_num + flag_cols + cat_cols
    col_order = [c for c in col_order if c in X_tr.columns]
    return X_tr[col_order], X_ot[col_order], col_medians, high_medians


# ── MASTER preprocessing (fit: train, transform: test) ────────────────────
X_train_ohe, X_test_ohe, ohe_enc, col_medians_low, col_medians_high = preprocess_ohe(
    X_train_raw, X_test_raw, cat_cols, low_nan_cols, high_nan_cols
)
X_train_raw_p, X_test_raw_p, _, _ = preprocess_raw(
    X_train_raw, X_test_raw, cat_cols, low_nan_cols, high_nan_cols,
    col_medians=col_medians_low, high_medians=col_medians_high
)

print(f'[OHE] X_train: {X_train_ohe.shape}   X_test: {X_test_ohe.shape}')
print(f'[RAW] X_train: {X_train_raw_p.shape}   X_test: {X_test_raw_p.shape}')
print(f'NaN kaldi mi (OHE train)? {X_train_ohe.isnull().any().any()}')
print(f'NaN kaldi mi (OHE test)?  {X_test_ohe.isnull().any().any()}')
print(f'NaN kaldi mi (RAW train)? {X_train_raw_p.isnull().any().any()}')
print(f'NaN kaldi mi (RAW test)?  {X_test_raw_p.isnull().any().any()}')

# is_missing flag sayisini raporla
flag_cols_in_ohe = [c for c in X_train_ohe.columns if c.startswith('is_missing_')]
print(f'\nis_missing flag sayisi (OHE matrisinde): {len(flag_cols_in_ohe)}')

[OHE] X_train: (2344, 618)   X_test: (587, 618)
[RAW] X_train: (2344, 514)   X_test: (587, 514)
NaN kaldi mi (OHE train)? False
NaN kaldi mi (OHE test)?  False
NaN kaldi mi (RAW train)? False
NaN kaldi mi (RAW test)?  False

is_missing flag sayisi (OHE matrisinde): 163


In [27]:
# Cell 5: Panel Verilerini Yukle & On Islemeyi Uygula
# Kritik: col_medians_low, col_medians_high ve ohe_enc MASTER train'den gelen degerler.
# Panellere fit yapilmaz, yalnizca transform uygulanir.

def preprocess_panel(path):
    df_p = pd.read_csv(path).drop(columns=['Variant_ID'], errors='ignore')
    y_p  = df_p['Label'].astype(int)
    X_p  = df_p.drop(columns=['Label'])

    # Eksik sutunlari NaN ile ekle (MASTER'da olan ama panelde olmayan)
    for c in X_master.drop(columns=['Label'], errors='ignore').columns:
        if c not in X_p.columns:
            X_p[c] = np.nan
    # Sadece MASTER sutunlariyla sinirla
    X_p = X_p[[c for c in X_master.drop(columns=['Label'], errors='ignore').columns
                if c in X_p.columns]]

    X_p_ohe, _, _, _, _ = preprocess_ohe(
        X_p, X_p,
        cat_cols, low_nan_cols, high_nan_cols,
        col_medians=col_medians_low, ohe=ohe_enc
    )
    X_p_raw, _, _, _ = preprocess_raw(
        X_p, X_p,
        cat_cols, low_nan_cols, high_nan_cols,
        col_medians=col_medians_low, high_medians=col_medians_high
    )
    return X_p_ohe, X_p_raw, y_p


panel_data = {}
for pname, path in PANEL_PATHS.items():
    X_p_ohe, X_p_raw, y_p = preprocess_panel(path)
    panel_data[pname] = (X_p_ohe, X_p_raw, y_p)
    vc = y_p.value_counts()
    print(f'{pname}: {len(y_p)} satir | pos={vc.get(1,0)} neg={vc.get(0,0)} '
          f'| OHE={X_p_ohe.shape[1]} RAW={X_p_raw.shape[1]}')

KANSER: 388 satir | pos=268 neg=120 | OHE=618 RAW=514
PAH: 372 satir | pos=310 neg=62 | OHE=618 RAW=514
CFTR: 111 satir | pos=90 neg=21 | OHE=618 RAW=514


In [28]:
# Cell 6: SECTION A - OHE Pipeline (LightGBM, XGBoost, SVM)
# ============================================================
# Kategorikler OHE ile binary sutunlara donusur.
# is_missing_* flagler sayisal sutun olarak matrise dahildir.
# Sayisal eksik degerler median ile doldurulmustur.

ohe_model_funcs = {
    'lgbm_ohe': (grid_search_lightgbm, 'lgbm'),
    'xgb_ohe':  (grid_search_xgboost,  'xgb'),
    'svm_ohe':  (grid_search_svm,       'svm'),
}

results_ohe = []
preds_ohe   = {}
trained_ohe = {}

for mt_key, (builder, short) in ohe_model_funcs.items():
    print(f"\n{'='*55}")
    print(f'  [OHE] MODEL: {mt_key.upper()}')
    print(f"{'='*55}")
    t0 = time.time()
    try:
        if short == 'svm':
            model, best_combo, best_thr, y_prob_ho, svm_scaler = builder(
                X_train_ohe, y_train, X_test_ohe, y_test, cat_features=[]
            )
            trained_ohe[mt_key] = {'model': model, 'threshold': best_thr,
                                   'svm_scaler': svm_scaler, 'short': short}
        else:
            model, best_combo, best_thr, y_prob_ho = builder(
                X_train_ohe, y_train, X_test_ohe, y_test, cat_features=[]
            )
            trained_ohe[mt_key] = {'model': model, 'threshold': best_thr, 'short': short}
    except Exception as e:
        print(f'  HATA: {e}'); continue

    elapsed   = time.time() - t0
    y_pred_ho = (y_prob_ho >= best_thr).astype(int)
    metrics   = compute_all_metrics(y_test, y_pred_ho, y_prob_ho)
    preds_ohe[mt_key] = {'y_pred': y_pred_ho, 'y_prob': y_prob_ho}
    results_ohe.append({
        'section': 'OHE', 'model': mt_key, 'threshold': best_thr,
        **metrics, 'elapsed_sec': round(elapsed, 1), 'best_params': str(best_combo),
        'n_train': len(y_train), 'n_test': len(y_test),
    })
    print(f'  F1={metrics["f1"]:.4f}  AUC={metrics["auc_roc"]:.4f}  '
          f'P={metrics["precision"]:.4f}  R={metrics["recall"]:.4f}  '
          f'MCC={metrics["mcc"]:.4f}  ({elapsed:.1f}s)')

results_ohe_df = pd.DataFrame(results_ohe)
print('\n=== SECTION A  OHE HOLD-OUT TABLOSU ===')
display(results_ohe_df[['model', 'f1', 'auc_roc', 'precision', 'recall', 'mcc', 'elapsed_sec']])


  [OHE] MODEL: LGBM_OHE
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 63, 'learning_rate': 0.1} -> CV F1=0.8818
  F1=0.8932  AUC=0.8334  P=0.8402  R=0.9535  MCC=0.5398  (23.5s)

  [OHE] MODEL: XGB_OHE
  XGBoost Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.1} -> CV F1=0.8827
  F1=0.8877  AUC=0.8388  P=0.8431  R=0.9372  MCC=0.5231  (11.0s)

  [OHE] MODEL: SVM_OHE
  SVM Grid Search: 6 kombinasyon
  En iyi combo: {'C': 0.1, 'gamma': 'auto'} -> CV F1=0.8697
  F1=0.8749  AUC=0.7949  P=0.8099  R=0.9512  MCC=0.4337  (33.4s)

=== SECTION A  OHE HOLD-OUT TABLOSU ===


,model,f1,auc_roc,precision,recall,mcc,elapsed_sec
0,lgbm_ohe,0.893246,0.833358,0.840164,0.953488,0.539836,23.5
1,xgb_ohe,0.887665,0.838809,0.843096,0.937209,0.523052,11.0
2,svm_ohe,0.874866,0.794912,0.809901,0.951163,0.433735,33.4


In [29]:
# Cell 7: SECTION B - LabelEncoder Pipeline (LightGBM, XGBoost, SVM)
# ====================================================================
# Kategorikler LabelEncoder ile integer'a donusur.
# is_missing_* flagler ve median-imputed sayisallar X_raw_p'de mevcut.
# Bilinmeyen paneli kategorileri -1 ile isaretlenir.

le_model_funcs = {
    'lgbm_le': (grid_search_le_lightgbm, 'lgbm'),
    'xgb_le':  (grid_search_le_xgboost,  'xgb'),
    'svm_le':  (grid_search_le_svm,       'svm'),
}

results_le = []
preds_le   = {}
trained_le = {}

for mt_key, (builder, short) in le_model_funcs.items():
    print(f"\n{'='*55}")
    print(f'  [LE] MODEL: {mt_key.upper()}')
    print(f"{'='*55}")
    t0 = time.time()
    try:
        result = builder(X_train_raw_p, y_train, X_test_raw_p, y_test, cat_cols=cat_cols)
        if short == 'svm':
            model, best_combo, best_thr, y_prob_ho, le_encoders, le_scaler = result
            trained_le[mt_key] = {'model': model, 'threshold': best_thr,
                                  'le_encoders': le_encoders, 'le_scaler': le_scaler, 'short': short}
        else:
            model, best_combo, best_thr, y_prob_ho, le_encoders = result
            trained_le[mt_key] = {'model': model, 'threshold': best_thr,
                                  'le_encoders': le_encoders, 'short': short}
    except Exception as e:
        print(f'  HATA: {e}'); continue

    elapsed   = time.time() - t0
    y_pred_ho = (y_prob_ho >= best_thr).astype(int)
    metrics   = compute_all_metrics(y_test, y_pred_ho, y_prob_ho)
    preds_le[mt_key] = {'y_pred': y_pred_ho, 'y_prob': y_prob_ho}
    results_le.append({
        'section': 'LE', 'model': mt_key, 'threshold': best_thr,
        **metrics, 'elapsed_sec': round(elapsed, 1), 'best_params': str(best_combo),
        'n_train': len(y_train), 'n_test': len(y_test),
    })
    print(f'  F1={metrics["f1"]:.4f}  AUC={metrics["auc_roc"]:.4f}  '
          f'P={metrics["precision"]:.4f}  R={metrics["recall"]:.4f}  '
          f'MCC={metrics["mcc"]:.4f}  ({elapsed:.1f}s)')

results_le_df = pd.DataFrame(results_le)
print('\n=== SECTION B  LE HOLD-OUT TABLOSU ===')
display(results_le_df[['model', 'f1', 'auc_roc', 'precision', 'recall', 'mcc', 'elapsed_sec']])


  [LE] MODEL: LGBM_LE
  LE+LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.05} -> CV F1=0.8807
  F1=0.8940  AUC=0.8415  P=0.8508  R=0.9419  MCC=0.5535  (22.3s)

  [LE] MODEL: XGB_LE
  LE+XGBoost Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.1} -> CV F1=0.8830
  F1=0.8832  AUC=0.8298  P=0.8191  R=0.9581  MCC=0.4785  (10.6s)

  [LE] MODEL: SVM_LE
  LE+SVM Grid Search: 4 kombinasyon
  En iyi combo: {'C': 1.0, 'gamma': 'auto'} -> CV F1=0.8679
  F1=0.8631  AUC=0.7810  P=0.7790  R=0.9674  MCC=0.3334  (19.7s)

=== SECTION B  LE HOLD-OUT TABLOSU ===


,model,f1,auc_roc,precision,recall,mcc,elapsed_sec
0,lgbm_le,0.894040,0.841520,0.850840,0.941860,0.553462,22.3
1,xgb_le,0.883173,0.829847,0.819085,0.958140,0.478466,10.6
2,svm_le,0.863071,0.780988,0.779026,0.967442,0.333370,19.7


In [30]:
# Cell 8: SECTION C - CatBoost (Ordered Target Statistics)
# =========================================================
# CatBoost kategorikleri native olarak alir.
# is_missing_* flagler sayisal olarak matriste; kategorikler ham string.
# Ordered Target Statistics (OTS): her kategorinin pathogenic orani.

print(f"\n{'='*55}")
print('  [CB] MODEL: CATBOOST')
print(f"{'='*55}")
t0 = time.time()

try:
    cb_model, cb_combo, cb_thr, cb_prob_ho = grid_search_catboost(
        X_train_raw_p, y_train, X_test_raw_p, y_test, cat_cols=cat_cols
    )

    elapsed    = time.time() - t0
    cb_pred_ho = (cb_prob_ho >= cb_thr).astype(int)
    cb_metrics = compute_all_metrics(y_test, cb_pred_ho, cb_prob_ho)

    preds_cb   = {'catboost': {'y_pred': cb_pred_ho, 'y_prob': cb_prob_ho}}
    trained_cb = {'catboost': {'model': cb_model, 'threshold': cb_thr}}

    results_cb_df = pd.DataFrame([{
        'section': 'CatBoost', 'model': 'catboost', 'threshold': cb_thr,
        **cb_metrics, 'elapsed_sec': round(elapsed, 1), 'best_params': str(cb_combo),
        'n_train': len(y_train), 'n_test': len(y_test),
    }])

    print(f'  F1={cb_metrics["f1"]:.4f}  AUC={cb_metrics["auc_roc"]:.4f}  '
          f'P={cb_metrics["precision"]:.4f}  R={cb_metrics["recall"]:.4f}  '
          f'MCC={cb_metrics["mcc"]:.4f}  ({elapsed:.1f}s)')
    print('\n=== SECTION C  CATBOOST HOLD-OUT TABLOSU ===')
    display(results_cb_df[['model', 'f1', 'auc_roc', 'precision', 'recall', 'mcc', 'elapsed_sec']])

except Exception as e:
    import traceback; traceback.print_exc()
    print(f'HATA: {e}')
    results_cb_df = pd.DataFrame()
    preds_cb      = {}
    trained_cb    = {}


  [CB] MODEL: CATBOOST
  CatBoost Grid Search: 12 kombinasyon  (cat_features=8)
  En iyi combo: {'iterations': 300, 'depth': 6, 'learning_rate': 0.1} -> CV F1=0.8870
  F1=0.8940  AUC=0.8364  P=0.8433  R=0.9512  MCC=0.5456  (51.5s)

=== SECTION C  CATBOOST HOLD-OUT TABLOSU ===


,model,f1,auc_roc,precision,recall,mcc,elapsed_sec
0,catboost,0.893989,0.83635,0.843299,0.951163,0.545645,51.5


In [31]:
# Cell 9: Sonuclari Birlestir + Panel Testi

results_all_df = pd.concat(
    [df for df in [results_ohe_df, results_le_df, results_cb_df] if len(df) > 0],
    ignore_index=True
)
results_all_df.to_csv(os.path.join(RESULTS_DIR, 'master_holdout_results.csv'), index=False)
print('=== MASTER HOLD-OUT  TUM MODELLER ===')
display(results_all_df[['section', 'model', 'f1', 'auc_roc', 'precision', 'recall', 'mcc', 'elapsed_sec']])

all_preds   = {**preds_ohe, **preds_le, **preds_cb}
all_trained = {**trained_ohe, **trained_le, **trained_cb}

model_pipeline = {}
for m in preds_ohe: model_pipeline[m] = 'ohe'
for m in preds_le:  model_pipeline[m] = 'le'
for m in preds_cb:  model_pipeline[m] = 'catboost'

# ── Panel Testi ───────────────────────────────────────────────────────────
panel_results   = []
panel_preds_all = {}


def _apply_le_encoders(X_panel, le_encoders):
    X_le = X_panel.copy()
    for col, le in le_encoders.items():
        if col in X_le.columns:
            known = set(le.classes_)
            X_le[col] = X_le[col].astype(str).apply(
                lambda v, k=known, e=le: e.transform([v])[0] if v in k else -1
            )
    return X_le


def _prep_catboost(X_panel, cat_cols):
    X_cb = X_panel.copy()
    for c in X_cb.columns:
        if c in cat_cols:
            X_cb[c] = X_cb[c].astype(str)
        else:
            X_cb[c] = pd.to_numeric(X_cb[c], errors='coerce').fillna(0).astype(float)
    return X_cb


for pname, (X_p_ohe, X_p_raw, y_p) in panel_data.items():
    print(f"\n{'='*55}")
    print(f'PANEL TEST: {pname}  ({len(y_p)} satir, pos={y_p.sum()}, neg={(y_p==0).sum()})')
    print(f"{'='*55}")
    panel_preds_all[pname] = {}

    for mt_name, info in all_trained.items():
        model    = info['model']
        best_thr = info['threshold']
        pipeline = model_pipeline[mt_name]
        X_panel  = X_p_ohe if pipeline == 'ohe' else X_p_raw

        try:
            if pipeline == 'ohe' and info.get('short') == 'svm':
                X_sc = info['svm_scaler'].transform(X_panel.fillna(0))
                y_prob_p = model.predict_proba(X_sc)[:, 1]
            elif pipeline == 'le' and info.get('short') == 'svm':
                X_le = _apply_le_encoders(X_panel, info['le_encoders'])
                X_sc = info['le_scaler'].transform(X_le.fillna(0))
                y_prob_p = model.predict_proba(X_sc)[:, 1]
            elif pipeline == 'le':
                X_le = _apply_le_encoders(X_panel, info['le_encoders'])
                y_prob_p = model.predict_proba(X_le)[:, 1]
            elif pipeline == 'catboost':
                X_cb = _prep_catboost(X_panel, cat_cols)
                y_prob_p = model.predict_proba(X_cb)[:, 1]
            else:
                y_prob_p = model.predict_proba(X_panel)[:, 1]

            y_pred_p  = (y_prob_p >= best_thr).astype(int)
            metrics_p = compute_all_metrics(y_p, y_pred_p, y_prob_p)
            panel_preds_all[pname][mt_name] = {'y_pred': y_pred_p, 'y_prob': y_prob_p}
            panel_results.append({
                'panel': pname, 'section': pipeline.upper(),
                'model': mt_name, 'threshold': best_thr, 'n_test': len(y_p),
                **metrics_p,
            })
            print(f'  [{pipeline.upper():7s}] {mt_name:14s}: '
                  f'F1={metrics_p["f1"]:.4f}  AUC={metrics_p["auc_roc"]:.4f}  '
                  f'R={metrics_p["recall"]:.4f}  P={metrics_p["precision"]:.4f}  '
                  f'MCC={metrics_p["mcc"]:.4f}')
        except Exception as e:
            print(f'  {mt_name}: HATA -- {e}')

panel_results_df = pd.DataFrame(panel_results)
panel_results_df.to_csv(os.path.join(RESULTS_DIR, 'panel_test_results.csv'), index=False)
print('\nPanel test sonuclari kaydedildi.')
display(panel_results_df[['panel', 'section', 'model', 'f1', 'auc_roc', 'recall', 'precision', 'mcc']])

=== MASTER HOLD-OUT  TUM MODELLER ===


,section,model,f1,auc_roc,precision,recall,mcc,elapsed_sec
0,OHE,lgbm_ohe,0.893246,0.833358,0.840164,0.953488,0.539836,23.5
1,OHE,xgb_ohe,0.887665,0.838809,0.843096,0.937209,0.523052,11.0
2,OHE,svm_ohe,0.874866,0.794912,0.809901,0.951163,0.433735,33.4
3,LE,lgbm_le,0.894040,0.841520,0.850840,0.941860,0.553462,22.3
4,LE,xgb_le,0.883173,0.829847,0.819085,0.958140,0.478466,10.6
5,LE,svm_le,0.863071,0.780988,0.779026,0.967442,0.333370,19.7
6,CatBoost,catboost,0.893989,0.836350,0.843299,0.951163,0.545645,51.5



PANEL TEST: KANSER  (388 satir, pos=268, neg=120)
  [OHE    ] lgbm_ohe      : F1=0.9016  AUC=0.8964  R=0.9739  P=0.8392  MCC=0.6457
  [OHE    ] xgb_ohe       : F1=0.8962  AUC=0.8884  R=0.9664  P=0.8355  MCC=0.6244
  [OHE    ] svm_ohe       : F1=0.8833  AUC=0.9023  R=0.9888  P=0.7982  MCC=0.5662
  [LE     ] lgbm_le       : F1=0.9072  AUC=0.9254  R=0.9851  P=0.8408  MCC=0.6687
  [LE     ] xgb_le        : F1=0.8829  AUC=0.9105  R=0.9851  P=0.8000  MCC=0.5640
  [LE     ] svm_le        : F1=0.8679  AUC=0.8917  R=0.9925  P=0.7710  MCC=0.4921
  [CATBOOST] catboost      : F1=0.9036  AUC=0.9000  R=0.9963  P=0.8266  MCC=0.6555

PANEL TEST: PAH  (372 satir, pos=310, neg=62)
  [OHE    ] lgbm_ohe      : F1=0.9136  AUC=0.8101  R=0.9548  P=0.8757  MCC=0.3588
  [OHE    ] xgb_ohe       : F1=0.9198  AUC=0.8046  R=0.9806  P=0.8661  MCC=0.3594
  [OHE    ] svm_ohe       : F1=0.8962  AUC=0.6399  R=0.9613  P=0.8394  MCC=0.0748
  [LE     ] lgbm_le       : F1=0.9294  AUC=0.8318  R=0.9774  P=0.8860  MCC=0.4768

,panel,section,model,f1,auc_roc,recall,precision,mcc
0,KANSER,OHE,lgbm_ohe,0.901554,0.896424,0.973881,0.839228,0.645735
1,KANSER,OHE,xgb_ohe,0.896194,0.888371,0.966418,0.835484,0.624399
2,KANSER,OHE,svm_ohe,0.883333,0.902285,0.988806,0.798193,0.566162
3,KANSER,LE,lgbm_le,0.907216,0.925435,0.985075,0.840764,0.668709
4,KANSER,LE,xgb_le,0.882943,0.910510,0.985075,0.800000,0.563963
5,KANSER,LE,svm_le,0.867863,0.891682,0.992537,0.771014,0.492069
6,KANSER,CATBOOST,catboost,0.903553,0.899969,0.996269,0.826625,0.655465
7,PAH,OHE,lgbm_ohe,0.913580,0.810094,0.954839,0.875740,0.358770
8,PAH,OHE,xgb_ohe,0.919818,0.804579,0.980645,0.866097,0.359419
9,PAH,OHE,svm_ohe,0.896241,0.639854,0.961290,0.839437,0.074838


In [32]:
# Cell 10: Cross-Model Error Analizi (MASTER Hold-Out)

error_df = pd.DataFrame(index=X_test_ohe.index)
error_df['y_true'] = y_test.values
for mt_name, preds in all_preds.items():
    error_df[f'{mt_name}_pred']  = preds['y_pred']
    error_df[f'{mt_name}_error'] = (preds['y_pred'] != y_test.values).astype(int)

error_cols = [c for c in error_df.columns if c.endswith('_error')]
error_df['difficulty_score'] = error_df[error_cols].sum(axis=1)
error_df.to_csv(os.path.join(RESULTS_DIR, 'error_analysis.csv'))

n_models = len(all_preds)
print(f'Hold-out test satirlari  : {len(error_df)}')
print(f'Tum modeller yanlis (score={n_models}): {(error_df["difficulty_score"]==n_models).sum()} satir')
print(f'Cogunluk yanlis (score>={n_models//2+1}): {(error_df["difficulty_score"]>=n_models//2+1).sum()} satir')
print(f'Tum modeller dogru (score=0): {(error_df["difficulty_score"]==0).sum()} satir')
print(f'\nDifficulty score dagilimi:')
print(error_df['difficulty_score'].value_counts().sort_index())

Hold-out test satirlari  : 587
Tum modeller yanlis (score=7): 61 satir
Cogunluk yanlis (score>=4): 104 satir
Tum modeller dogru (score=0): 414 satir

Difficulty score dagilimi:
difficulty_score
0    414
1     43
2     16
3     10
4     13
5     13
6     17
7     61
Name: count, dtype: int64


In [33]:
# Cell 11: Gorsellesme
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig_paths = []

flat_colors = {
    'lgbm_ohe': '#2E7D32', 'xgb_ohe': '#1565C0', 'svm_ohe': '#E65100',
    'lgbm_le':  '#66BB6A', 'xgb_le':  '#42A5F5', 'svm_le':  '#FFA726',
    'catboost': '#F44336',
}
model_order = list(results_all_df['model']) if len(results_all_df) > 0 else []

# 1. MASTER Hold-Out: F1 ve AUC-ROC
if len(results_all_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, metric, title in zip(
        axes, ['f1', 'auc_roc'], ['Hold-Out F1 Score', 'Hold-Out AUC-ROC']
    ):
        colors = [flat_colors.get(m, '#607D8B') for m in results_all_df['model']]
        bars   = ax.bar(results_all_df['model'], results_all_df[metric], color=colors)
        ax.bar_label(bars, fmt='%.4f', fontsize=7)
        ax.set_title(f'{title} -- MASTER (is_missing + Median Imputation)', fontsize=10)
        ax.set_ylim(0, 1.14)
        ax.tick_params(axis='x', rotation=30)
        for patch, (_, row) in zip(bars, results_all_df.iterrows()):
            ax.text(patch.get_x() + patch.get_width()/2,
                    patch.get_height() + 0.047,
                    row['section'], ha='center', fontsize=6, color='#555')
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'master_f1_auc.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 2. Tam metrik karsilastirma
if len(results_all_df) > 0:
    metrics_to_plot = ['f1', 'auc_roc', 'precision', 'recall', 'balanced_accuracy']
    n_m   = len(results_all_df)
    width = 0.8 / n_m
    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(metrics_to_plot))
    for i, (_, row) in enumerate(results_all_df.iterrows()):
        bars = ax.bar(x + i * width, [row[m] for m in metrics_to_plot], width,
                      label=f"[{row['section']}] {row['model']}",
                      color=flat_colors.get(row['model'], '#607D8B'))
        ax.bar_label(bars, fmt='%.3f', fontsize=5, padding=1)
    ax.set_xticks(x + width * (n_m - 1) / 2)
    ax.set_xticklabels(metrics_to_plot, fontsize=9)
    ax.set_title('MASTER Hold-Out -- Tam Metrik Karsilastirma (is_missing + Median)', fontsize=10)
    ax.set_ylim(0, 1.30); ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'master_metrics.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 3. Panel F1 heatmap
if len(panel_results_df) > 0:
    pivot_f1 = panel_results_df.pivot(index='panel', columns='model', values='f1')
    ordered  = [m for m in model_order if m in pivot_f1.columns]
    fig, ax  = plt.subplots(figsize=(14, 4))
    sns.heatmap(pivot_f1[ordered], annot=True, fmt='.4f', cmap='YlGn',
                vmin=0.7, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title('Panel Test F1 Heatmap -- 3 Pipeline x 3 Panel', fontsize=11)
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'panel_f1_heatmap.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 4. Panel Recall heatmap
if len(panel_results_df) > 0:
    pivot_rec = panel_results_df.pivot(index='panel', columns='model', values='recall')
    ordered   = [m for m in model_order if m in pivot_rec.columns]
    fig, ax   = plt.subplots(figsize=(14, 4))
    sns.heatmap(pivot_rec[ordered], annot=True, fmt='.4f', cmap='Blues',
                vmin=0.7, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title('Panel Test Recall Heatmap -- Klinik Kritik Metrik', fontsize=11)
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'panel_recall_heatmap.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 5. LightGBM OHE Feature Importance
if 'lgbm_ohe' in trained_ohe:
    lgbm_obj = trained_ohe['lgbm_ohe']['model']
    fi_df = pd.DataFrame({
        'feature': X_train_ohe.columns, 'importance': lgbm_obj.feature_importances_
    }).sort_values('importance', ascending=False).head(30)
    flag_in_top = [f for f in fi_df['feature'] if f.startswith('is_missing_')]
    print(f'Top-30 ozellik icerisinde is_missing flag sayisi: {len(flag_in_top)}')
    if flag_in_top:
        print(f'  -> {flag_in_top}')
    fig, ax = plt.subplots(figsize=(10, 10))
    colors_fi = ['#7B1FA2' if f.startswith('is_missing_') else '#2E7D32'
                 for f in fi_df['feature'].values[::-1]]
    ax.barh(range(len(fi_df)), fi_df['importance'].values[::-1], color=colors_fi)
    ax.set_yticks(range(len(fi_df)))
    ax.set_yticklabels(fi_df['feature'].values[::-1], fontsize=8)
    ax.set_title('LightGBM [OHE] -- Top 30 Feature Importance\n(Mor = is_missing flag)', fontsize=10)
    ax.set_xlabel('Importance (Split)')
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'lgbm_ohe_feature_importance.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 6. CatBoost Feature Importance
if 'catboost' in trained_cb:
    cb_obj = trained_cb['catboost']['model']
    fi_cb  = pd.DataFrame({
        'feature': cb_obj.feature_names_,
        'importance': cb_obj.get_feature_importance()
    }).sort_values('importance', ascending=False).head(30)
    fig, ax = plt.subplots(figsize=(10, 10))
    colors_cb = ['#7B1FA2' if f.startswith('is_missing_') else '#F44336'
                 for f in fi_cb['feature'].values[::-1]]
    ax.barh(range(len(fi_cb)), fi_cb['importance'].values[::-1], color=colors_cb)
    ax.set_yticks(range(len(fi_cb)))
    ax.set_yticklabels(fi_cb['feature'].values[::-1], fontsize=8)
    ax.set_title('CatBoost [OTS] -- Top 30 Feature Importance\n(Mor = is_missing flag)', fontsize=10)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, 'catboost_feature_importance.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 7. Confusion Matrix -- her model
for mt_name, preds in all_preds.items():
    cm = confusion_matrix(y_test, preds['y_pred'])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax, cmap='Blues')
    sec = model_pipeline.get(mt_name, '').upper()
    ax.set_title(f'[{sec}] {mt_name.upper()}', fontsize=10)
    plt.tight_layout()
    p = os.path.join(RESULTS_DIR, f'master_{mt_name}_cm.png')
    fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 8. Panel x Model Confusion Matrix
if len(panel_results_df) > 0:
    panels_list = list(panel_data.keys())
    for mt_name in all_trained.keys():
        if mt_name not in panel_preds_all.get(panels_list[0], {}):
            continue
        fig, axes = plt.subplots(1, len(panels_list), figsize=(5 * len(panels_list), 4.2))
        if len(panels_list) == 1:
            axes = [axes]
        sec = model_pipeline.get(mt_name, '').upper()
        fig.suptitle(f'[{sec}] {mt_name.upper()} -- Panel Bazli Confusion Matrix',
                     fontsize=12, fontweight='bold')
        for ax, pname in zip(axes, panels_list):
            preds_p = panel_preds_all[pname].get(mt_name)
            if preds_p is None:
                ax.set_title(f'{pname} (veri yok)'); ax.axis('off'); continue
            y_p    = panel_data[pname][2]
            y_pred = preds_p['y_pred']
            cm     = confusion_matrix(y_p, y_pred, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            total  = tn + fp + fn + tp
            acc    = (tn + tp) / total if total > 0 else 0
            ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(
                ax=ax, cmap='Blues', colorbar=False, values_format='d'
            )
            ax.set_title(
                f'{pname}  (n={total})\nAcc={acc:.3f}  FP={fp}  FN={fn}', fontsize=9
            )
            ax.set_xlabel('Tahmin'); ax.set_ylabel('Gercek')
        plt.tight_layout()
        p = os.path.join(RESULTS_DIR, f'panel_cm_{mt_name}.png')
        fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

# 9. Difficulty Score
fig, ax = plt.subplots(figsize=(8, 5))
error_df['difficulty_score'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color='#9C27B0', edgecolor='white')
ax.set_title(f'Difficulty Score (0=Tum Dogru, {n_models}=Tum Yanlis)', fontsize=11)
ax.set_xlabel('Difficulty Score'); ax.set_ylabel('Satir Sayisi')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', fontsize=9)
plt.tight_layout()
p = os.path.join(RESULTS_DIR, 'difficulty_distribution.png')
fig.savefig(p, dpi=150); fig_paths.append(p); plt.close()

print(f'\nToplam {len(fig_paths)} grafik kaydedildi.')

Top-30 ozellik icerisinde is_missing flag sayisi: 0

Toplam 21 grafik kaydedildi.


In [34]:
# Cell 12: PDF Rapor Olusturma
from fpdf import FPDF
from datetime import datetime


class BaselineReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 9)
        self.cell(0, 7, 'Teknofest - NB12 Baseline (is_missing + Median Imputation)',
                  align='C', new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(2)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}/{{nb}}', align='C')

    def sec_title(self, txt):
        self.set_font('Helvetica', 'B', 13)
        self.cell(0, 10, txt, new_x='LMARGIN', new_y='NEXT'); self.ln(1)

    def sub_title(self, txt):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 7, txt, new_x='LMARGIN', new_y='NEXT')

    def body(self, txt):
        self.set_font('Helvetica', '', 9)
        self.cell(0, 6, txt, new_x='LMARGIN', new_y='NEXT')

    def th(self, cols, widths):
        self.set_font('Helvetica', 'B', 8)
        for w, h in zip(widths, cols):
            self.cell(w, 7, h, border=1, align='C')
        self.ln()

    def tr(self, vals, widths):
        self.set_font('Helvetica', '', 8)
        for w, v in zip(widths, vals):
            self.cell(w, 6, v, border=1, align='C')
        self.ln()


pdf = BaselineReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Baslik
pdf.add_page()
pdf.set_font('Helvetica', 'B', 18)
pdf.ln(18)
pdf.cell(0, 12, 'Gercek Veri Baseline Egitim Raporu', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 13)
pdf.cell(0, 8, 'is_missing flags + Median Imputation -- 3 Pipeline, 7 Model',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(6)
pdf.set_font('Helvetica', '', 10)
for line in [
    f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    'Egitim verisi: YARISMA_TRAIN_MASTER.csv',
    'Test panelleri: KANSER, PAH, CFTR',
    'Pipeline A (OHE)     : LightGBM, XGBoost, SVM',
    'Pipeline B (LE)      : LightGBM, XGBoost, SVM',
    'Pipeline C (CatBoost): CatBoost (Ordered Target Statistics)',
    f'Ham veri: {X_master.shape[0]} satir x {X_master.shape[1]+1} sutun (Label dahil)',
    f'Yuksek NaN (>%50) sutunlar: {len(high_nan_cols)} -> is_missing flag olarak tutuldu',
    f'Dusuk NaN (<=50%) sutunlar: {len(low_nan_cols)} -> MASTER train mediani ile dolduruldu',
]:
    pdf.cell(0, 7, line, align='C', new_x='LMARGIN', new_y='NEXT')

# 1. On Isleme Aciklamasi
pdf.add_page()
pdf.sec_title('1. On Isleme Stratejisi')
pdf.sub_title('NaN Yonetimi')
pdf.set_font('Helvetica', '', 9)
pdf.multi_cell(0, 5,
    'Klasik %50 NaN drop stratejisi terk edildi. '
    'CLAUDE.md\'a gore eksiklik bilgi tasiyor: '
    'MASTER\'da Label=1 satirlarda eksiklik %59.9, Label=0\'da %41.2. '
    'Bu fark, eksikligi bir sinyal olarak modele sunmanin gecerliligini gosteriyor. '
    'NaN>%50 olan sutunlar drop edilmek yerine is_missing_* binary flag olarak tutuluyor; '
    'sutunun orijinal degeri de (varsa) ayni sekilde matriste kaliyor. '
    'NaN>%50 olan sutunun kendisi icin de median imputation uygulanir (model NaN kabul etmez).')
pdf.ln(2)
pdf.sub_title('Median vs Mean Imputation')
pdf.set_font('Helvetica', '', 9)
pdf.multi_cell(0, 5,
    'AL_ sutunlari populasyon freakans ve patojenite skoru icerir. '
    'Bu dagilimlar genellikle saga carpik (cogu varyant nadir). '
    'Mean imputation, carpik dagilimda aykiri gozlemlere karsi hassastir. '
    'Median bu etkiye karsi direncli oldugu icin tercih edildi.')
pdf.ln(2)
pdf.sub_title('Leakage Korumalari')
pdf.set_font('Helvetica', '', 9)
pdf.multi_cell(0, 5,
    'Tum fit islemleri (median hesabi, OHE, LabelEncoder) YALNIZCA MASTER train bolumu uzerinde yapildi. '
    'Test seti ve panellere yalnizca transform uygulanir. '
    'Panel testlerinde MASTER train\'dan cikarilan median ve encoderlar kullanilir.')
pdf.ln(3)

# 2. Veri Ozeti
pdf.add_page()
pdf.sec_title('2. Veri Ozeti')
for line in [
    f'  MASTER ham boyut: {X_master.shape[0]} satir x {X_master.shape[1]+1} sutun',
    f'  Label: Pozitif={int(y_master.sum())}, Negatif={int((y_master==0).sum())} (oran={y_master.mean():.3f})',
    f'  Kategorik sutunlar ({len(cat_cols)}): {", ".join(cat_cols)}',
    f'  Sayisal sutun toplam: {len(num_cols)}',
    f'    NaN > %50 -> is_missing flag: {len(high_nan_cols)}',
    f'    NaN <= %50 -> median imputed: {len(low_nan_cols)}',
    f'  OHE boyutu: {X_train_ohe.shape[1]} ozellik',
    f'  RAW boyutu: {X_train_raw_p.shape[1]} ozellik',
    f'  Hold-out: egitim={len(X_train_raw)}, test={len(X_test_raw)}',
]:
    pdf.body(line)
pdf.ln(3)
pdf.sub_title('Panel Verileri:')
pw = [35, 22, 22, 22, 50]
pdf.th(['Panel', 'Toplam', 'Pozitif', 'Negatif', 'MASTER Ortusme'], pw)
master_raw = pd.read_csv(MASTER_PATH)
master_ids = set(master_raw['Variant_ID'])
for pname, path in PANEL_PATHS.items():
    df_praw = pd.read_csv(path)
    vc = df_praw['Label'].value_counts()
    overlap = len(set(df_praw['Variant_ID']) & master_ids)
    pdf.tr([
        pname, str(len(df_praw)), str(vc.get(1, 0)), str(vc.get(0, 0)),
        f'{overlap}/{len(df_praw)} ({overlap/len(df_praw)*100:.0f}%)',
    ], pw)

# 3. MASTER Hold-Out
pdf.add_page()
pdf.sec_title('3. MASTER Hold-Out Sonuclari (%20 hold-out)')
cw = [24, 13, 13, 13, 13, 13, 13, 13, 10]
for lbl, df_sec in [
    ('Pipeline A -- OHE', results_ohe_df),
    ('Pipeline B -- LE',  results_le_df),
    ('Pipeline C -- CatBoost', results_cb_df),
]:
    if len(df_sec) == 0:
        continue
    pdf.sub_title(lbl)
    pdf.th(['Model', 'F1', 'AUC-ROC', 'AUC-PR', 'Prec', 'Recall', 'MCC', 'Bal.Acc', 'Sec'], cw)
    for _, row in df_sec.iterrows():
        pdf.tr([
            row['model'],
            f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}", f"{row['auc_pr']:.4f}",
            f"{row['precision']:.4f}", f"{row['recall']:.4f}",
            f"{row['mcc']:.4f}", f"{row['balanced_accuracy']:.4f}",
            f"{row['elapsed_sec']:.0f}",
        ], cw)
    pdf.ln(3)

if len(results_all_df) > 0:
    best_row = results_all_df.loc[results_all_df['f1'].idxmax()]
    pdf.sub_title('En Iyi Model (F1 bazinda):')
    pdf.body(
        f"  [{best_row['section']}] {best_row['model']} --"
        f" F1={best_row['f1']:.4f}  AUC={best_row['auc_roc']:.4f}"
        f"  Prec={best_row['precision']:.4f}  Recall={best_row['recall']:.4f}"
    )

# 4. Panel Test
if len(panel_results_df) > 0:
    pdf.add_page()
    pdf.sec_title('4. Panel Test Sonuclari')
    pdf.set_font('Helvetica', 'I', 9)
    pdf.cell(0, 6,
             'Her panel dosyasinin tum satirlari test edildi. '
             'MASTER train median ve encoder kullanildi (fit yok).',
             new_x='LMARGIN', new_y='NEXT')
    pdf.ln(2)
    cw2 = [18, 12, 20, 13, 13, 13, 13, 13, 7]
    pdf.th(['Panel', 'Sec', 'Model', 'F1', 'AUC-ROC', 'Prec', 'Recall', 'MCC', 'N'], cw2)
    for _, row in panel_results_df.sort_values(['panel', 'section', 'model']).iterrows():
        pdf.tr([
            row['panel'], row['section'], row['model'],
            f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}",
            f"{row['precision']:.4f}", f"{row['recall']:.4f}",
            f"{row['mcc']:.4f}", str(row['n_test']),
        ], cw2)
    pdf.ln(3)
    pdf.sub_title('En Iyi Model (Panel Bazinda):')
    for pname in panel_results_df['panel'].unique():
        sub  = panel_results_df[panel_results_df['panel'] == pname]
        best = sub.loc[sub['f1'].idxmax()]
        pdf.body(
            f"  {pname}: [{best['section']}] {best['model']} --"
            f" F1={best['f1']:.4f}  Recall={best['recall']:.4f}  AUC={best['auc_roc']:.4f}"
        )

# 5. Error Analizi
pdf.add_page()
pdf.sec_title('5. Cross-Model Error Analizi')
n_m  = len(all_preds)
diff = error_df['difficulty_score']
for line in [
    f'Hold-out satirlari: {len(error_df)}',
    f'Tum modeller yanlis (score={n_m}): {(diff == n_m).sum()} satir',
    f'Cogunluk yanlis (score>={n_m//2+1}): {(diff >= n_m//2+1).sum()} satir',
    f'Tum modeller dogru (score=0): {(diff == 0).sum()} satir',
]:
    pdf.body(line)
pdf.ln(2)
pdf.sub_title('Difficulty Score Dagilimi:')
for score, count in diff.value_counts().sort_index().items():
    pdf.body(f'  Score={score}: {count} satir')

# Grafikler
for fp in fig_paths:
    if os.path.exists(fp):
        pdf.add_page()
        fname = os.path.basename(fp).replace('.png', '').replace('_', ' ').title()
        pdf.set_font('Helvetica', 'B', 11)
        pdf.cell(0, 9, fname, new_x='LMARGIN', new_y='NEXT')
        try:
            pdf.image(fp, x=10, w=190)
        except Exception as e:
            pdf.body(f'Grafik yuklenemedi: {e}')

report_path = os.path.join(REPORTS_DIR, 'real_data_baseline_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/real_data_baseline_report.pdf
